# Calvin Site 1 and SO4/T4 parameter exploration

Calvin combines core root length density and dry mass with recovered Site 1 structure. SO4/T4
uses eight digitized 18-month vines in the v2 soil frame. Both models use the 0.93 mm local
parent branching threshold and the 2.5 mm diameter reporting split. The notebook reads saved
trial records by default. The replay cell resumes trials from a matching engine snapshot.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import os
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "deeprootgen").is_dir())
study = root / ".runtime/structural-search-20261118"
bundle = root / "Calvin_SO4_Parameter_Exploration_2026-11-18"
previous_calvin = root / ".runtime/structural-starts-20261118/Calvin"
read = lambda p: json.loads(p.read_text())
digest = lambda p: sha256(p.read_bytes()).hexdigest()
assert study.is_dir() and bundle.is_dir()
try:
    display
except NameError:
    from IPython.display import display
print("Working checkout:", root)

## Site 1 structural evidence

The standing vine supplies the primary count, diameter groups and emergence depths. Pulled
vines supply basal diameters on recovered segments. Recovery favored thick roots. Recorded
order rises at both forks and side branches. Segment lengths provide approximate first-branch
priors. Side-branch attachment positions and chronological ages are unresolved. The 250 cm
length limit is a researcher-specified contour-length prior. The core observations retain
scanned length from 0.93 to 2.5 mm and dry mass at diameters of at least 0.93 mm.

In [ ]:
prior = read(study / "Calvin/site1_priors.private.json")
counts = pd.DataFrame(prior["pulled_recorded_segment_counts_by_rep_and_order"]).T.rename_axis("REP")
print("Standing vine primaries:", prior["field_primary_count"])
print("Thicker primaries by the 8.2 mm exposed-diameter split:", prior["field_thick_count_at_least_8_2_mm"])
display(counts)
summary = []
for order, row in prior["orders"].items():
    d = row["start_diameter_mm"]
    l = row["recorded_branched_segment_length_cm"]
    summary.append({"Recorded order":order, "Valid basal diameters":d["n"], "Mean basal diameter (mm)":d["mean"],
                    "Branched segment mean length (cm)":l.get("mean"),
                    "Unbroken two-daughter median (cm)":row["unbroken_two_daughter_interval_cm"].get("median")})
display(pd.DataFrame(summary).round(3))
k = prior["taper"]["k_segments_10cm_or_longer"]
print(f'Pooled k = {k["median"]:.6f}/cm (n = {k["n"]}, IQR {k["q25"]:.6f}–{k["q75"]:.6f}/cm)')

The exponential taper uses d(s) = d₀ exp(−ks) until a root narrows to 4 mm.
A continuous thin taper then supplies the remaining profile. The point diameter controls
branching and capping at each attachment. Exported dry mass integrates squared diameter along
the geometry. Core projections use the same material law on the clipped support. A unit-density
run supplies the dry-mass predictions for the full density grid through linear conversion.

In [ ]:
distance = np.linspace(0, 250, 251)
fig, ax = plt.subplots(figsize=(7,3.2), constrained_layout=True)
for order, row in prior["orders"].items():
    d = row["start_diameter_mm"]["median"] * np.exp(-k["median"] * distance)
    keep = d >= 4.
    ax.plot(distance[keep], d[keep], label=f"Recorded order {order}")
ax.axhline(4., color="#777777", lw=1, ls="--", label="4 mm transition")
ax.set(xlabel="Distance along a root (cm)", ylabel="Diameter (mm)",
       title="Thick-root exponential taper from the pooled Site 1 rate")
ax.legend(frameon=False, fontsize=8)
display(fig)
plt.close(fig)

## Replay or resume a search

Fill the tuple in the next cell to run a named study. An empty tuple keeps this an inspection
notebook. Resume only after the matching process has exited. The launchers use the qualified
Python 3.10 environment and the source snapshot recorded in each protocol.

In [ ]:
resume = ()
launchers = {
    "SO4": ("source-before", "SO4/search_so4.py", ("--workers", "9")),
    "SO4-axial": ("source-final", "SO4-axial/search_axial.py", ("--workers", "10")),
    "SO4-final": ("source-final", "SO4-final/search_final.py", ("--workers", "9")),
    "Calvin-final": ("source-final", "Calvin-final/finalize.py", ("--workers", "11")),
    "Calvin-qc": ("source-final", "Calvin-qc/check_all.py", ("--workers", "15")),
    "Calvin": ("source-taper", "Calvin/search_calvin.py", ("--workers", "13")),
}
for name in resume:
    snapshot, script, args = launchers[name]
    env = dict(os.environ)
    env.update(PYTHONPATH=os.pathsep.join(map(str, (study / snapshot,
        root / ".runtime/native-integration-20261117/locked-overlay", root))),
        PYTHONDONTWRITEBYTECODE="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1",
        OPENBLAS_NUM_THREADS="1", CUDA_VISIBLE_DEVICES="")
    python = root / ".runtime/envs/py310-qualification/bin/python"
    subprocess.run([str(python), str(study / script), *args], env=env, cwd=root, check=True)

## Try a manual parameter change

Add a dictionary to the list to test a different value without overwriting a selected input.
The example uses the Site 1 input and varies its first-order length. Name each trial and choose
a seed. The launcher records the complete configuration, scores, diagnostics and core projections.
Empty the list again to inspect results without running a new simulation.

In [ ]:
manual_trials = [
    # {"dataset":"Calvin", "name":"longer_first_order", "seed":25001,
    #  "config":str(bundle / "Calvin_Site1_start.schema4.private.json"),
    #  "changes":{"features.lengths.settings.by_order.1.mean":90.0}},
]
for trial in manual_trials:
    import tempfile
    env = dict(os.environ)
    env.update(PYTHONPATH=os.pathsep.join(map(str, (
        root / ".runtime/native-integration-20261117/locked-overlay", root))),
        PYTHONDONTWRITEBYTECODE="1", OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
        MKL_NUM_THREADS="1", CUDA_VISIBLE_DEVICES="")
    python = root / ".runtime/envs/py310-qualification/bin/python"
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json") as request:
        json.dump(trial,request)
        request.flush()
        subprocess.run([str(python),str(study / "manual_trial.py"),request.name],
                       env=env,cwd=root,check=True)

## Run ledger

Each record points to its saved receipt. SO4 settings and the starter hash reconstruct each
trial. Calvin receipts link to the full schema-4 input and case-level core predictions
at unit density. A partial run may contain a parameters file before it has a receipt.
The ledger includes failed simulations and their recorded reasons.

In [ ]:
study_dirs = [p for p in sorted(study.iterdir()) if (p / "runs").is_dir()]
records = []
for folder in study_dirs:
    for path in sorted((folder / "runs").rglob("*.json")):
        if path.name in ("parameters.json", "manifest.json"):
            continue
        row = read(path)
        if not {"id","seed"}.issubset(row):
            continue
        failure = row.get("error")
        invalid = row.get("invalid", 0)
        records.append({"study":folder.name, "candidate":row["id"], "seed":row["seed"],
                        "resolution":"4 cm on orders 4–5" if row.get("screening") else "full",
                        "status":"passed" if failure is None and invalid == 0 else "failed",
                        "reason":failure or (f"{invalid} invalid roots" if invalid else ""),
                        "receipt":str(path.relative_to(root)),
                        "parameters":json.dumps(row.get("params", row.get("parameters"))),
                        "root_length_cm":row.get("metrics", {}).get("soil_length_cm"),
                        "longest_primary_cm":row.get("max_primary_cm"),
                        "run_identity":row.get("run_identity", row.get("run_identity_sha256"))})
    for path in sorted((folder / "runs").rglob("parameters.json")):
        if (path.parent / "receipt.json").is_file():
            continue
        p = read(path)
        records.append({"study":folder.name, "candidate":path.parent.parent.name,
                        "seed":p["random_seed"], "resolution":"full",
                        "status":"awaiting receipt",
                        "reason":"Configuration saved before the run receipt",
                        "receipt":str(path.relative_to(root))})
ledger = pd.DataFrame(records)
export_tables = False
if export_tables:
    ledger.to_csv(bundle / "Trial_ledger.csv", index=False)
display(ledger.groupby(["study","resolution","status"]).size().unstack(fill_value=0))

In [ ]:
score_rows = []
for folder in study_dirs:
    for path in sorted(folder.glob("*.json")):
        source = read(path)
        if not isinstance(source, list) or not source or "id" not in source[0]:
            continue
        for item in source:
            if "metrics" not in item and "scored" not in item and "error" not in item:
                continue
            score_rows.append({"study":folder.name, "stage":path.stem, "candidate":item["id"],
                               "parameters":json.dumps(item.get("settings", item.get("parameters"))),
                               "metrics":json.dumps(item.get("metrics", item.get("scored"))),
                               "error":json.dumps(item.get("error")),
                               "record":str(path.relative_to(root))})
if export_tables:
    pd.DataFrame(score_rows).to_csv(bundle / "Candidate_scores.csv", index=False)
print("Candidate-by-stage score rows:", len(score_rows))

## Confirmation records

SO4 has sixteen relative errors for population means and three 10 cm marginal length
profile total-variation comparisons. The eight observed vines informed the length
distributions. New seeds test simulation variability on that same observed support.
Calvin compares predictions on the eleven fitting and three retained Site 1 core locations.
Separate RLD and mass metrics show 20% mean targets, profile residuals and the effect of
measurement bounds. Flagged rows and the external Site 5 check remain visible.

In [ ]:
for folder in study_dirs:
    path = folder / "confirmation.json"
    if not path.is_file():
        continue
    table = []
    for row in read(path):
        if row.get("metrics"):
            m = row["metrics"]
            table.append({"candidate":row["id"], "seeds":m["n"], "within 20%":m["within_20_percent"],
                          "max error":m["max_error"], "across-row span bias":m["relative_errors"]["across_span_cm"],
                          "along-row span bias":m["relative_errors"]["along_span_cm"], "depth TV":m["profile_tv"][2]})
        for fold, score in row.get("scored",{}).items():
            for metric, values in score["metrics"].items():
                table.append({"candidate":row["id"], "support":fold, "quantity":metric,
                              "density (g/cm³)":score["density_g_cm3"], **values})
    print(folder.name)
    if table:
        display(pd.DataFrame(table).round(3))

In [ ]:
combined_file = study / "Calvin_combined_confirmation.json"
if combined_file.is_file():
    combined = read(combined_file)
    chosen = combined["model_choice"]
    metrics = combined["models"][chosen]
    quality_rows = []
    for scope in ("Site1_hold", "Site1_hold_qc", "Site1_all", "Site5_external"):
        for quantity, values in metrics[scope]["metrics"].items():
            quality_rows.append({"support":scope, "quantity":quantity, **values})
    print("Selected Calvin input, pooled predictions over",
          sum(map(len,combined["seeds"])), "reserved seeds")
    display(pd.DataFrame(quality_rows).round(3))

In [ ]:
for path in sorted(bundle.glob("*.private.json")):
    print(path.name, digest(path)[:16])